In [1]:
%pip install requests beautifulsoup4 lxml pandas numpy tqdm sentence-transformers faiss-cpu

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import json
import requests
import numpy as np
import pandas as pd

from pathlib import Path
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

from sentence_transformers import SentenceTransformer
import faiss

print("RAG environment ready!")

c:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


RAG environment ready!


In [3]:
import requests
import bs4
import lxml
import pandas
import numpy
import tqdm
import sentence_transformers
import faiss

print("requests:", requests.__version__)
print("beautifulsoup4:", bs4.__version__)
print("lxml:", lxml.__version__)
print("pandas:", pandas.__version__)
print("numpy:", numpy.__version__)
print("sentence-transformers:", sentence_transformers.__version__)
print("FAISS: imported successfully")

print("\n✅ All RAG dependencies are ready!")

requests: 2.34.2
beautifulsoup4: 4.15.0
lxml: 6.1.3
pandas: 3.0.6
numpy: 2.4.6
sentence-transformers: 6.1.0
FAISS: imported successfully

✅ All RAG dependencies are ready!


In [4]:
import json
import os

rag_sources = [
    {
        "id": "radiologyinfo_chest_xray_report",
        "title": "How to Read Your Chest X-ray Report",
        "organization": "RadiologyInfo",
        "url": "https://www.radiologyinfo.org/en/info/article-chest-xray-report",
        "type": "medical_reference"
    },
    {
        "id": "chestxray14_research",
        "title": "ChestX-ray14 Dataset and Multi-Label Chest X-ray Classification",
        "organization": "NIH / Research Literature",
        "url": "https://pmc.ncbi.nlm.nih.gov/articles/PMC9131331/",
        "type": "research"
    }
]

os.makedirs("rag/metadata", exist_ok=True)

with open(
    "rag/metadata/sources.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(rag_sources, f, indent=4)

print("RAG source registry created!")
print("Sources:", len(rag_sources))

for source in rag_sources:
    print(f"✓ {source['title']}")

RAG source registry created!
Sources: 2
✓ How to Read Your Chest X-ray Report
✓ ChestX-ray14 Dataset and Multi-Label Chest X-ray Classification


In [5]:
import requests

pmc_id = "PMC9131331"

url = (
    "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"
    f"efetch.fcgi?db=pmc&id={pmc_id}&retmode=xml"
)

response = requests.get(url, timeout=60)

print("HTTP status:", response.status_code)
print("Response size:", len(response.text), "characters")
print("Starts with:", response.text[:100])


HTTP status: 200
Response size: 11528 characters
Starts with: <?xml version="1.0"  ?><!DOCTYPE pmc-articleset PUBLIC "-//NLM//DTD ARTICLE SET 2.0//EN" "https://dt


In [6]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(response.text, "xml")

# Extract the article text
text_parts = []

for element in soup.find_all(["article-title", "abstract", "p"]):
    text = element.get_text(" ", strip=True)
    if text:
        text_parts.append(text)

article_text = "\n\n".join(text_parts)

print("Extracted article text:", len(article_text), "characters")
print("\nFirst 2000 characters:\n")
print(article_text[:2000])

Extracted article text: 4637 characters

First 2000 characters:

Modeling long-range dependencies for weakly supervised disease classification and localization on chest X-ray

Contributions: (I) Conception and design: All authors; (II) Administrative support: All authors; (III) Provision of study materials or patients: C Chen, S Yang; (IV) Collection and assembly of data: F Li; (V) Data analysis and interpretation: F Li; (VI) Manuscript writing: All authors; (VII) Final approval of manuscript: All authors.

These authors contributed equally to this work and should be considered as co-first authors.

Conflicts of Interest: All authors have completed the ICMJE uniform disclosure form (available at https://qims.amegroups.com/article/view/10.21037/qims-21-1117/coif ). FL has a pending patent (No. 202110640995.8). The other authors have no conflicts of interest to declare.

Background Computer-aided diagnosis based on chest X-ray (CXR) is an exponentially growing field of research owing to 

In [8]:
import os

print("Current working directory:")
print(os.getcwd())

print("\nContents:")
print(os.listdir())

Current working directory:
c:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\notebooks

Contents:
['Chest_Xray_AI.ipynb', 'rag', 'RAG_LLM.ipynb']


In [10]:
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()

RAG_DIR = PROJECT_ROOT / "rag"
DOCUMENTS_DIR = RAG_DIR / "documents"
CHUNKS_DIR = RAG_DIR / "chunks"
VECTOR_STORE_DIR = RAG_DIR / "vector_store"
METADATA_DIR = RAG_DIR / "metadata"

for directory in [
    RAG_DIR,
    DOCUMENTS_DIR,
    CHUNKS_DIR,
    VECTOR_STORE_DIR,
    METADATA_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("RAG directory:", RAG_DIR)
print("Documents directory:", DOCUMENTS_DIR)
print("✓ RAG paths ready!")

Project root: C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI
RAG directory: C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag
Documents directory: C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\documents
✓ RAG paths ready!


In [11]:
document_path = DOCUMENTS_DIR / "chestxray14_research.txt"

with open(
    document_path,
    "w",
    encoding="utf-8"
) as f:
    f.write(article_text)

print("Document saved successfully!")
print("Characters:", len(article_text))
print("Path:", document_path)

Document saved successfully!
Characters: 4637
Path: C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\documents\chestxray14_research.txt


In [12]:
# Inspect the document structure

with open(
    document_path,
    "r",
    encoding="utf-8"
) as f:
    text = f.read()

paragraphs = [
    p.strip()
    for p in text.split("\n\n")
    if p.strip()
]

print("Total characters:", len(text))
print("Total paragraphs:", len(paragraphs))

print("\nFirst 10 paragraphs:\n")

for i, paragraph in enumerate(paragraphs[:10]):
    print(f"\n--- Paragraph {i + 1} ---")
    print(paragraph[:500])

Total characters: 4637
Total paragraphs: 9

First 10 paragraphs:


--- Paragraph 1 ---
Modeling long-range dependencies for weakly supervised disease classification and localization on chest X-ray

--- Paragraph 2 ---
Contributions: (I) Conception and design: All authors; (II) Administrative support: All authors; (III) Provision of study materials or patients: C Chen, S Yang; (IV) Collection and assembly of data: F Li; (V) Data analysis and interpretation: F Li; (VI) Manuscript writing: All authors; (VII) Final approval of manuscript: All authors.

--- Paragraph 3 ---
These authors contributed equally to this work and should be considered as co-first authors.

--- Paragraph 4 ---
Conflicts of Interest: All authors have completed the ICMJE uniform disclosure form (available at https://qims.amegroups.com/article/view/10.21037/qims-21-1117/coif ). FL has a pending patent (No. 202110640995.8). The other authors have no conflicts of interest to declare.

--- Paragraph 5 ---
Background Compu

In [13]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(response.text, "xml")

title = soup.find("article-title")
title_text = title.get_text(" ", strip=True) if title else ""

abstract = soup.find("abstract")
abstract_text = abstract.get_text(" ", strip=True) if abstract else ""

# Extract paragraph-level content only
paragraphs = []

for p in soup.find_all("p"):
    text = p.get_text(" ", strip=True)
    if text:
        paragraphs.append(text)

# Build clean document
clean_article_text = "\n\n".join(
    [title_text, abstract_text] + paragraphs
)

print("Cleaned document characters:", len(clean_article_text))
print("Paragraphs:", len(paragraphs))

print("\nFirst 2000 characters:\n")
print(clean_article_text[:2000])

Cleaned document characters: 4637
Paragraphs: 7

First 2000 characters:

Modeling long-range dependencies for weakly supervised disease classification and localization on chest X-ray

Background Computer-aided diagnosis based on chest X-ray (CXR) is an exponentially growing field of research owing to the development of deep learning, especially convolutional neural networks (CNNs). However, due to the intrinsic locality of convolution operations, CNNs cannot model long-range dependencies. Although vision transformers (ViTs) have recently been proposed to alleviate this limitation, those trained on patches cannot learn any dependencies for inter-patch pixels and thus, are insufficient for medical image detection. To address this problem, in this paper, we propose a CXR detection method which integrates CNN with a ViT for modeling patch-wise and inter-patch dependencies. Methods We experimented on the ChestX-ray14 dataset and followed the official training-test set split. Because the tra

In [14]:
document_path = DOCUMENTS_DIR / "chestxray14_research.txt"

with open(
    document_path,
    "w",
    encoding="utf-8"
) as f:
    f.write(clean_article_text)

print("Clean document saved!")
print("Characters:", len(clean_article_text))
print("Path:", document_path)

Clean document saved!
Characters: 4637
Path: C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\documents\chestxray14_research.txt


In [15]:
chunk_size = 800
chunk_overlap = 150

chunks = []

start = 0
text_length = len(clean_article_text)

while start < text_length:
    end = min(start + chunk_size, text_length)

    chunk = clean_article_text[start:end].strip()

    if chunk:
        chunks.append(chunk)

    start += chunk_size - chunk_overlap

print("Document characters:", text_length)
print("Number of chunks:", len(chunks))

print("\nChunk sizes:")
for i, chunk in enumerate(chunks):
    print(f"Chunk {i + 1}: {len(chunk)} characters")

print("\nFirst chunk:\n")
print(chunks[0])

Document characters: 4637
Number of chunks: 8

Chunk sizes:
Chunk 1: 800 characters
Chunk 2: 800 characters
Chunk 3: 799 characters
Chunk 4: 799 characters
Chunk 5: 800 characters
Chunk 6: 799 characters
Chunk 7: 737 characters
Chunk 8: 87 characters

First chunk:

Modeling long-range dependencies for weakly supervised disease classification and localization on chest X-ray

Background Computer-aided diagnosis based on chest X-ray (CXR) is an exponentially growing field of research owing to the development of deep learning, especially convolutional neural networks (CNNs). However, due to the intrinsic locality of convolution operations, CNNs cannot model long-range dependencies. Although vision transformers (ViTs) have recently been proposed to alleviate this limitation, those trained on patches cannot learn any dependencies for inter-patch pixels and thus, are insufficient for medical image detection. To address this problem, in this paper, we propose a CXR detection method which integ

In [16]:
MIN_CHUNK_SIZE = 200

clean_chunks = []

for chunk in chunks:
    chunk = chunk.strip()

    if not chunk:
        continue

    if len(chunk) < MIN_CHUNK_SIZE and clean_chunks:
        clean_chunks[-1] += " " + chunk
    else:
        clean_chunks.append(chunk)

chunks = clean_chunks

print("Final number of chunks:", len(chunks))

for i, chunk in enumerate(chunks):
    print(f"Chunk {i + 1}: {len(chunk)} characters")

Final number of chunks: 7
Chunk 1: 800 characters
Chunk 2: 800 characters
Chunk 3: 799 characters
Chunk 4: 799 characters
Chunk 5: 800 characters
Chunk 6: 799 characters
Chunk 7: 825 characters


In [17]:
chunk_records = []

for i, chunk in enumerate(chunks):
    chunk_records.append({
        "chunk_id": f"chestxray14_{i+1:03d}",
        "source_id": "chestxray14_research",
        "source_title": "ChestX-ray14 Dataset and Multi-Label Chest X-ray Classification",
        "source_url": "https://pmc.ncbi.nlm.nih.gov/articles/PMC9131331/",
        "chunk_index": i,
        "text": chunk
    })

chunks_path = CHUNKS_DIR / "chestxray14_chunks.json"

with open(chunks_path, "w", encoding="utf-8") as f:
    json.dump(chunk_records, f, indent=2, ensure_ascii=False)

print("Saved:", chunks_path)
print("Number of chunk records:", len(chunk_records))
print("\nFirst chunk metadata:")
print(json.dumps(chunk_records[0], indent=2, ensure_ascii=False))

Saved: C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\chunks\chestxray14_chunks.json
Number of chunk records: 7

First chunk metadata:
{
  "chunk_id": "chestxray14_001",
  "source_id": "chestxray14_research",
  "source_title": "ChestX-ray14 Dataset and Multi-Label Chest X-ray Classification",
  "source_url": "https://pmc.ncbi.nlm.nih.gov/articles/PMC9131331/",
  "chunk_index": 0,
  "text": "Modeling long-range dependencies for weakly supervised disease classification and localization on chest X-ray\n\nBackground Computer-aided diagnosis based on chest X-ray (CXR) is an exponentially growing field of research owing to the development of deep learning, especially convolutional neural networks (CNNs). However, due to the intrinsic locality of convolution operations, CNNs cannot model long-range dependencies. Although vision transformers (ViTs) have recently been proposed to alleviate this limitation, those trained on patches cannot learn any dependencies for inter-patch pixels and t

In [18]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully.")
print("Embedding dimension:", embedding_model.get_sentence_embedding_dimension())

c:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\yuvan\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\.venv\Lib\site-packages\hu

Embedding model loaded successfully.
Embedding dimension: 384


C:\Users\yuvan\AppData\Local\Temp\ipykernel_36920\3015113085.py:4: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:", embedding_model.get_sentence_embedding_dimension())


In [19]:
texts = [record["text"] for record in chunk_records]

embeddings = embedding_model.encode(
    texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Embeddings shape:", embeddings.shape)
print("Embedding dtype:", embeddings.dtype)
print("First embedding (first 10 values):")
print(embeddings[0][:10])

Batches: 100%|██████████| 1/1 [00:00<00:00,  1.83it/s]

Embeddings shape: (7, 384)
Embedding dtype: float32
First embedding (first 10 values):
[-0.00103039 -0.01264188  0.07845823 -0.06923418  0.08762156 -0.02444632
 -0.04653818  0.0073574  -0.03253129 -0.1186422 ]


In [20]:
embedding_dim = embeddings.shape[1]

index = faiss.IndexFlatIP(embedding_dim)
index.add(embeddings)

print("FAISS index created successfully.")
print("Embedding dimension:", embedding_dim)
print("Vectors stored in index:", index.ntotal)

FAISS index created successfully.
Embedding dimension: 384
Vectors stored in index: 7


In [21]:
vector_store_path = VECTOR_STORE_DIR / "chestxray14.index"

faiss.write_index(index, str(vector_store_path))

print("FAISS index saved successfully.")
print("Path:", vector_store_path)
print("File size (KB):", round(vector_store_path.stat().st_size / 1024, 2))

FAISS index saved successfully.
Path: C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\vector_store\chestxray14.index
File size (KB): 10.54


In [22]:
metadata_records = [
    {
        "faiss_index": record["chunk_index"],
        "chunk_id": record["chunk_id"],
        "source_id": record["source_id"],
        "source_title": record["source_title"],
        "source_url": record["source_url"]
    }
    for record in chunk_records
]

metadata_path = METADATA_DIR / "chestxray14_metadata.json"

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata_records, f, indent=2, ensure_ascii=False)

print("Metadata saved successfully.")
print("Path:", metadata_path)
print("Records:", len(metadata_records))
print("\nFirst record:")
print(json.dumps(metadata_records[0], indent=2, ensure_ascii=False))

Metadata saved successfully.
Path: C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\metadata\chestxray14_metadata.json
Records: 7

First record:
{
  "faiss_index": 0,
  "chunk_id": "chestxray14_001",
  "source_id": "chestxray14_research",
  "source_title": "ChestX-ray14 Dataset and Multi-Label Chest X-ray Classification",
  "source_url": "https://pmc.ncbi.nlm.nih.gov/articles/PMC9131331/"
}


In [23]:
query = "What is the ChestX-ray14 dataset?"

query_embedding = embedding_model.encode(
    [query],
    convert_to_numpy=True,
    normalize_embeddings=True
)

top_k = 3
scores, indices = index.search(query_embedding, top_k)

print("Query:", query)
print("\nRetrieved chunks:\n")

for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):
    record = chunk_records[idx]

    print(f"--- Rank {rank} | Score: {score:.4f} ---")
    print("Chunk ID:", record["chunk_id"])
    print(record["text"][:500])
    print()

Query: What is the ChestX-ray14 dataset?

Retrieved chunks:

--- Rank 1 | Score: 0.4504 ---
Chunk ID: chestxray14_002
o address this problem, in this paper, we propose a CXR detection method which integrates CNN with a ViT for modeling patch-wise and inter-patch dependencies. Methods We experimented on the ChestX-ray14 dataset and followed the official training-test set split. Because the training data only had global annotations, the detection network was weakly supervised. A DenseNet with a feature pyramid structure was designed and integrated with an adaptive ViT to model inter-patch and patch-wise long-rang

--- Rank 2 | Score: 0.4503 ---
Chunk ID: chestxray14_005
cle/view/10.21037/qims-21-1117/coif ). FL has a pending patent (No. 202110640995.8). The other authors have no conflicts of interest to declare.

Computer-aided diagnosis based on chest X-ray (CXR) is an exponentially growing field of research owing to the development of deep learning, especially convolutional neural netw

In [24]:
for i, record in enumerate(chunk_records):
    print("=" * 80)
    print(f"CHUNK {i + 1} | {record['chunk_id']} | {len(record['text'])} characters")
    print("=" * 80)
    print(record["text"])
    print()

CHUNK 1 | chestxray14_001 | 800 characters
Modeling long-range dependencies for weakly supervised disease classification and localization on chest X-ray

Background Computer-aided diagnosis based on chest X-ray (CXR) is an exponentially growing field of research owing to the development of deep learning, especially convolutional neural networks (CNNs). However, due to the intrinsic locality of convolution operations, CNNs cannot model long-range dependencies. Although vision transformers (ViTs) have recently been proposed to alleviate this limitation, those trained on patches cannot learn any dependencies for inter-patch pixels and thus, are insufficient for medical image detection. To address this problem, in this paper, we propose a CXR detection method which integrates CNN with a ViT for modeling patch-wise and inter-patch depen

CHUNK 2 | chestxray14_002 | 800 characters
o address this problem, in this paper, we propose a CXR detection method which integrates CNN with a ViT for mod

In [25]:
sections = soup.find_all("sec")

print("Number of sections:", len(sections))

for i, sec in enumerate(sections):
    title_tag = sec.find("title", recursive=False)
    title_text = title_tag.get_text(" ", strip=True) if title_tag else "[No title]"
    print(f"{i + 1}. {title_text}")

Number of sections: 4
1. Background
2. Methods
3. Results
4. Conclusions


In [26]:
clean_sections = []

for sec in soup.find_all("sec", recursive=False):
    title_tag = sec.find("title", recursive=False)
    
    if not title_tag:
        continue
    
    section_title = title_tag.get_text(" ", strip=True)
    
    paragraphs = []
    for p in sec.find_all("p"):
        text = p.get_text(" ", strip=True)
        if text:
            paragraphs.append(text)
    
    section_text = "\n\n".join(paragraphs).strip()
    
    if section_text:
        clean_sections.append({
            "section": section_title,
            "text": section_text
        })

print("Clean sections extracted:", len(clean_sections))

for section in clean_sections:
    print(f"\n{'=' * 70}")
    print(section["section"])
    print(f"{'=' * 70}")
    print("Characters:", len(section["text"]))
    print(section["text"][:500])

Clean sections extracted: 0


In [27]:
clean_sections = []

for sec in soup.find_all("sec"):
    title_tag = sec.find("title", recursive=False)

    if not title_tag:
        continue

    section_title = title_tag.get_text(" ", strip=True)

    # Only keep the four main article sections
    if section_title not in ["Background", "Methods", "Results", "Conclusions"]:
        continue

    paragraphs = []
    for p in sec.find_all("p"):
        text = p.get_text(" ", strip=True)
        if text:
            paragraphs.append(text)

    section_text = "\n\n".join(paragraphs).strip()

    if section_text:
        clean_sections.append({
            "section": section_title,
            "text": section_text
        })

print("Clean sections extracted:", len(clean_sections))

for section in clean_sections:
    print(f"\n{'=' * 70}")
    print(section["section"])
    print(f"{'=' * 70}")
    print("Characters:", len(section["text"]))
    print(section["text"][:500])

Clean sections extracted: 4

Background
Characters: 686
Computer-aided diagnosis based on chest X-ray (CXR) is an exponentially growing field of research owing to the development of deep learning, especially convolutional neural networks (CNNs). However, due to the intrinsic locality of convolution operations, CNNs cannot model long-range dependencies. Although vision transformers (ViTs) have recently been proposed to alleviate this limitation, those trained on patches cannot learn any dependencies for inter-patch pixels and thus, are insufficient fo

Methods
Characters: 476
We experimented on the ChestX-ray14 dataset and followed the official training-test set split. Because the training data only had global annotations, the detection network was weakly supervised. A DenseNet with a feature pyramid structure was designed and integrated with an adaptive ViT to model inter-patch and patch-wise long-range dependencies and obtain fine-grained feature maps. We compared the performance usin

In [28]:
semantic_chunks = []

for i, section in enumerate(clean_sections):
    semantic_chunks.append({
        "chunk_id": f"chestxray14_{i+1:03d}",
        "source_id": "chestxray14_research",
        "source_title": "ChestX-ray14 Dataset and Multi-Label Chest X-ray Classification",
        "source_url": "https://pmc.ncbi.nlm.nih.gov/articles/PMC9131331/",
        "section": section["section"],
        "chunk_index": i,
        "text": section["text"]
    })

print("Semantic chunks:", len(semantic_chunks))

for chunk in semantic_chunks:
    print(
        f"{chunk['chunk_id']} | "
        f"{chunk['section']} | "
        f"{len(chunk['text'])} characters"
    )

Semantic chunks: 4
chestxray14_001 | Background | 686 characters
chestxray14_002 | Methods | 476 characters
chestxray14_003 | Results | 542 characters
chestxray14_004 | Conclusions | 181 characters


In [29]:
chunk_records = semantic_chunks

chunks_path = CHUNKS_DIR / "chestxray14_chunks.json"

with open(chunks_path, "w", encoding="utf-8") as f:
    json.dump(chunk_records, f, indent=2, ensure_ascii=False)

print("Clean chunk file updated successfully.")
print("Path:", chunks_path)
print("Number of chunks:", len(chunk_records))

Clean chunk file updated successfully.
Path: C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\chunks\chestxray14_chunks.json
Number of chunks: 4


In [30]:
texts = [record["text"] for record in chunk_records]

embeddings = embedding_model.encode(
    texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("New embeddings shape:", embeddings.shape)
print("Embedding dtype:", embeddings.dtype)

Batches: 100%|██████████| 1/1 [00:00<00:00, 18.39it/s]

New embeddings shape: (4, 384)
Embedding dtype: float32


In [31]:
embedding_dim = embeddings.shape[1]

index = faiss.IndexFlatIP(embedding_dim)
index.add(embeddings)

print("FAISS index rebuilt successfully.")
print("Embedding dimension:", embedding_dim)
print("Vectors stored in index:", index.ntotal)

FAISS index rebuilt successfully.
Embedding dimension: 384
Vectors stored in index: 4


In [32]:
vector_store_path = VECTOR_STORE_DIR / "chestxray14.index"

faiss.write_index(index, str(vector_store_path))

print("Updated FAISS index saved successfully.")
print("Path:", vector_store_path)
print("Vectors stored:", index.ntotal)
print("File size (KB):", round(vector_store_path.stat().st_size / 1024, 2))

Updated FAISS index saved successfully.
Path: C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\vector_store\chestxray14.index
Vectors stored: 4
File size (KB): 6.04


In [33]:
metadata_records = [
    {
        "faiss_index": record["chunk_index"],
        "chunk_id": record["chunk_id"],
        "source_id": record["source_id"],
        "source_title": record["source_title"],
        "source_url": record["source_url"],
        "section": record["section"]
    }
    for record in chunk_records
]

metadata_path = METADATA_DIR / "chestxray14_metadata.json"

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata_records, f, indent=2, ensure_ascii=False)

print("Metadata updated successfully.")
print("Records:", len(metadata_records))

for record in metadata_records:
    print(
        f"{record['faiss_index']} → "
        f"{record['chunk_id']} → "
        f"{record['section']}"
    )

Metadata updated successfully.
Records: 4
0 → chestxray14_001 → Background
1 → chestxray14_002 → Methods
2 → chestxray14_003 → Results
3 → chestxray14_004 → Conclusions


In [34]:
query = "What is the ChestX-ray14 dataset?"

query_embedding = embedding_model.encode(
    [query],
    convert_to_numpy=True,
    normalize_embeddings=True
)

top_k = 3
scores, indices = index.search(query_embedding, top_k)

print("Query:", query)
print("\nRetrieved chunks:\n")

for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):
    record = chunk_records[idx]

    print(f"--- Rank {rank} | Score: {score:.4f} ---")
    print("Chunk ID:", record["chunk_id"])
    print("Section:", record["section"])
    print(record["text"])
    print()

Query: What is the ChestX-ray14 dataset?

Retrieved chunks:

--- Rank 1 | Score: 0.5667 ---
Chunk ID: chestxray14_002
Section: Methods
We experimented on the ChestX-ray14 dataset and followed the official training-test set split. Because the training data only had global annotations, the detection network was weakly supervised. A DenseNet with a feature pyramid structure was designed and integrated with an adaptive ViT to model inter-patch and patch-wise long-range dependencies and obtain fine-grained feature maps. We compared the performance using our method with that of other disease detection methods.

--- Rank 2 | Score: 0.4160 ---
Chunk ID: chestxray14_001
Section: Background
Computer-aided diagnosis based on chest X-ray (CXR) is an exponentially growing field of research owing to the development of deep learning, especially convolutional neural networks (CNNs). However, due to the intrinsic locality of convolution operations, CNNs cannot model long-range dependencies. Although vi

In [35]:
rag_sources = [
    {
        "id": "chestxray14_dataset",
        "title": "NIH ChestX-ray14 Dataset",
        "organization": "NIH Clinical Center / Research Literature",
        "url": "https://pmc.ncbi.nlm.nih.gov/articles/PMC9131331/",
        "type": "dataset_research",
        "purpose": "Dataset characteristics, 14 findings, multi-label classification, and label generation"
    },
    {
        "id": "radiologyinfo_chest_xray",
        "title": "Chest X-ray (Radiography)",
        "organization": "RadiologyInfo",
        "url": "https://www.radiologyinfo.org/en/info/chestrad",
        "type": "medical_reference",
        "purpose": "Chest X-ray overview, uses, procedure, interpretation context, and limitations"
    },
    {
        "id": "radiologyinfo_chest_xray_report",
        "title": "How to Read Your Chest X-ray Report",
        "organization": "RadiologyInfo",
        "url": "https://www.radiologyinfo.org/en/info/article-chest-xray-report",
        "type": "medical_reference",
        "purpose": "Radiology report terminology, findings, impression, and common chest X-ray findings"
    }
]

sources_path = METADATA_DIR / "sources.json"

with open(sources_path, "w", encoding="utf-8") as f:
    json.dump(rag_sources, f, indent=2, ensure_ascii=False)

print("Source registry updated successfully.")
print("Number of sources:", len(rag_sources))

for source in rag_sources:
    print(f"{source['id']} → {source['title']}")

Source registry updated successfully.
Number of sources: 3
chestxray14_dataset → NIH ChestX-ray14 Dataset
radiologyinfo_chest_xray → Chest X-ray (Radiography)
radiologyinfo_chest_xray_report → How to Read Your Chest X-ray Report


In [36]:
pmc_id = "PMC9131331"

url = (
    "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"
    f"efetch.fcgi?db=pmc&id={pmc_id}&retmode=xml"
)

response = requests.get(url, timeout=60)

print("HTTP status:", response.status_code)
print("Downloaded characters:", len(response.text))
print("Content starts with:", response.text[:100])

HTTP status: 200
Downloaded characters: 11528
Content starts with: <?xml version="1.0"  ?><!DOCTYPE pmc-articleset PUBLIC "-//NLM//DTD ARTICLE SET 2.0//EN" "https://dt


In [37]:
source_soup = BeautifulSoup(response.text, "xml")

target_sections = ["Background", "Methods", "Results", "Conclusions"]

source_sections = []

for sec in source_soup.find_all("sec"):
    title_tag = sec.find("title", recursive=False)

    if not title_tag:
        continue

    section_title = title_tag.get_text(" ", strip=True)

    if section_title not in target_sections:
        continue

    paragraphs = []

    for p in sec.find_all("p"):
        text = p.get_text(" ", strip=True)
        if text:
            paragraphs.append(text)

    section_text = "\n\n".join(paragraphs).strip()

    if section_text:
        source_sections.append({
            "section": section_title,
            "text": section_text
        })

print("Sections extracted:", len(source_sections))

for section in source_sections:
    print(
        f"{section['section']}: "
        f"{len(section['text'])} characters"
    )

Sections extracted: 4
Background: 686 characters
Methods: 476 characters
Results: 542 characters
Conclusions: 181 characters


In [38]:
nih_document = "\n\n".join(
    f"{section['section']}\n\n{section['text']}"
    for section in source_sections
)

nih_path = DOCUMENTS_DIR / "chestxray14_nih_research.txt"

with open(nih_path, "w", encoding="utf-8") as f:
    f.write(nih_document)

print("NIH research document saved successfully.")
print("Path:", nih_path)
print("Characters:", len(nih_document))

NIH research document saved successfully.
Path: C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\documents\chestxray14_nih_research.txt
Characters: 1934


In [39]:
radiology_url = "https://www.radiologyinfo.org/en/info/chestrad"

response_radiology = requests.get(
    radiology_url,
    timeout=60,
    headers={
        "User-Agent": "Mozilla/5.0"
    }
)

print("HTTP status:", response_radiology.status_code)
print("Downloaded characters:", len(response_radiology.text))
print("Content starts with:", response_radiology.text[:100])

HTTP status: 200
Downloaded characters: 55182
Content starts with: 


<!doctype html>
<html lang="en">

<head>
<meta charset="utf-8" />
<title>Chest X-ray (Rad


In [40]:
radiology_soup = BeautifulSoup(response_radiology.text, "lxml")

headings = radiology_soup.find_all(["h1", "h2", "h3"])

print("Number of headings:", len(headings))

for i, heading in enumerate(headings[:40]):
    text = heading.get_text(" ", strip=True)
    if text:
        print(f"{i + 1}. {heading.name}: {text}")

Number of headings: 18
1. h1: Chest X-ray
2. h2: What is a chest X-ray?
3. h2: What are some common uses of the procedure?
4. h2: How should I prepare?
5. h2: What does the equipment look like?
6. h2: How does the procedure work?
7. h2: How is the procedure performed?
8. h2: What will I experience during and after the procedure?
9. h2: Who interprets the results and how do I get them?
10. h2: What are the benefits vs. risks?
11. h3: Benefits
12. h3: Risks
13. h3: A Word About Minimizing Radiation Exposure
14. h2: What are the limitations of chest x-ray?
15. h2: Additional Information and Resources
16. h2: Send us your feedback
17. h2: Image Gallery
18. h1: Glossary


In [41]:
target_headings = [
    "What is a chest X-ray?",
    "What are some common uses of the procedure?",
    "How should I prepare?",
    "What does the equipment look like?",
    "How does the procedure work?",
    "How is the procedure performed?",
    "What will I experience during and after the procedure?",
    "Who interprets the results and how do I get them?",
    "What are the benefits vs. risks?",
    "What are the limitations of chest x-ray?"
]

radiology_sections = []

for heading in radiology_soup.find_all(["h2", "h3"]):
    heading_text = heading.get_text(" ", strip=True)

    if heading_text not in target_headings:
        continue

    content_parts = []
    
    for element in heading.find_all_next():
        if element.name in ["h2", "h1"] and element is not heading:
            break
        
        if element.name in ["p", "li"]:
            text = element.get_text(" ", strip=True)
            if text:
                content_parts.append(text)

    section_text = "\n\n".join(content_parts).strip()

    if section_text:
        radiology_sections.append({
            "section": heading_text,
            "text": section_text
        })

print("Sections extracted:", len(radiology_sections))

for section in radiology_sections:
    print(
        f"{section['section']} → "
        f"{len(section['text'])} characters"
    )

Sections extracted: 10
What is a chest X-ray? → 437 characters
What are some common uses of the procedure? → 526 characters
How should I prepare? → 627 characters
What does the equipment look like? → 700 characters
How does the procedure work? → 996 characters
How is the procedure performed? → 1157 characters
What will I experience during and after the procedure? → 461 characters
Who interprets the results and how do I get them? → 693 characters
What are the benefits vs. risks? → 1446 characters
What are the limitations of chest x-ray? → 552 characters


In [42]:
radiology_document = "\n\n".join(
    f"{section['section']}\n\n{section['text']}"
    for section in radiology_sections
)

radiology_path = DOCUMENTS_DIR / "radiologyinfo_chest_xray.txt"

with open(radiology_path, "w", encoding="utf-8") as f:
    f.write(radiology_document)

print("RadiologyInfo document saved successfully.")
print("Path:", radiology_path)
print("Characters:", len(radiology_document))

RadiologyInfo document saved successfully.
Path: C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\documents\radiologyinfo_chest_xray.txt
Characters: 7987


In [43]:
report_url = "https://www.radiologyinfo.org/en/info/article-chest-xray-report"

response_report = requests.get(
    report_url,
    timeout=60,
    headers={
        "User-Agent": "Mozilla/5.0"
    }
)

print("HTTP status:", response_report.status_code)
print("Downloaded characters:", len(response_report.text))
print("Content starts with:", response_report.text[:100])

HTTP status: 200
Downloaded characters: 45585
Content starts with: 


<!doctype html>
<html lang="en">

<head>
<meta charset="utf-8" />
<title>How to Read Your


In [44]:
report_url = "https://www.radiologyinfo.org/en/info/article-chest-xray-report"

response_report = requests.get(
    report_url,
    timeout=60,
    headers={
        "User-Agent": "Mozilla/5.0"
    }
)

print("HTTP status:", response_report.status_code)
print("Downloaded characters:", len(response_report.text))
print("Content starts with:", response_report.text[:100])

HTTP status: 200
Downloaded characters: 45585
Content starts with: 


<!doctype html>
<html lang="en">

<head>
<meta charset="utf-8" />
<title>How to Read Your


In [45]:
report_soup = BeautifulSoup(response_report.text, "lxml")

headings = report_soup.find_all(["h1", "h2", "h3"])

print("Number of headings:", len(headings))

for i, heading in enumerate(headings[:40]):
    text = heading.get_text(" ", strip=True)
    if text:
        print(f"{i + 1}. {heading.name}: {text}")

Number of headings: 13
1. h1: How to Read Your Chest X-ray Report
2. h2: What is Chest X-ray commonly used for?
3. h2: Sections of the Radiology Report
4. h3: Type of exam
5. h3: History/Reason for exam
6. h3: Comparison/Priors
7. h3: Technique
8. h3: Findings
9. h3: Impression
10. h2: Additional Information
11. h2: Send us your feedback
12. h2: Image Gallery
13. h1: Glossary


In [46]:
target_report_headings = [
    "What is Chest X-ray commonly used for?",
    "Sections of the Radiology Report",
    "Type of exam",
    "History/Reason for exam",
    "Comparison/Priors",
    "Technique",
    "Findings",
    "Impression",
    "Additional Information"
]

report_sections = []

for heading in report_soup.find_all(["h2", "h3"]):
    heading_text = heading.get_text(" ", strip=True)

    if heading_text not in target_report_headings:
        continue

    content_parts = []

    for element in heading.find_all_next():
        if element.name in ["h2", "h1"] and element is not heading:
            break

        if element.name in ["p", "li"]:
            text = element.get_text(" ", strip=True)
            if text:
                content_parts.append(text)

    section_text = "\n\n".join(content_parts).strip()

    if section_text:
        report_sections.append({
            "section": heading_text,
            "text": section_text
        })

print("Sections extracted:", len(report_sections))

for section in report_sections:
    print(
        f"{section['section']} → "
        f"{len(section['text'])} characters"
    )

Sections extracted: 9
What is Chest X-ray commonly used for? → 434 characters
Sections of the Radiology Report → 4131 characters
Type of exam → 4131 characters
History/Reason for exam → 3963 characters
Comparison/Priors → 3345 characters
Technique → 2579 characters
Findings → 2237 characters
Impression → 1381 characters
Additional Information → 594 characters


In [47]:
report_sections = []

all_headings = report_soup.find_all(["h1", "h2", "h3"])

for heading in all_headings:
    heading_text = heading.get_text(" ", strip=True)

    if heading_text not in target_report_headings:
        continue

    current_level = int(heading.name[1])
    content_parts = []

    for element in heading.find_all_next():
        if element.name in ["h1", "h2", "h3"]:
            next_level = int(element.name[1])

            if next_level <= current_level:
                break

        if element.name in ["p", "li"]:
            text = element.get_text(" ", strip=True)
            if text:
                content_parts.append(text)

    section_text = "\n\n".join(content_parts).strip()

    if section_text:
        report_sections.append({
            "section": heading_text,
            "level": current_level,
            "text": section_text
        })

print("Sections extracted:", len(report_sections))

for section in report_sections:
    print(
        f"{section['section']} "
        f"(h{section['level']}) → "
        f"{len(section['text'])} characters"
    )

Sections extracted: 9
What is Chest X-ray commonly used for? (h2) → 434 characters
Sections of the Radiology Report (h2) → 4131 characters
Type of exam (h3) → 166 characters
History/Reason for exam (h3) → 616 characters
Comparison/Priors (h3) → 764 characters
Technique (h3) → 340 characters
Findings (h3) → 854 characters
Impression (h3) → 1381 characters
Additional Information (h2) → 594 characters


In [48]:
report_sections_clean = [
    section for section in report_sections
    if section["section"] != "Sections of the Radiology Report"
]

report_document = "\n\n".join(
    f"{section['section']}\n\n{section['text']}"
    for section in report_sections_clean
)

report_path = DOCUMENTS_DIR / "radiologyinfo_chest_xray_report.txt"

with open(report_path, "w", encoding="utf-8") as f:
    f.write(report_document)

print("RadiologyInfo report document saved successfully.")
print("Path:", report_path)
print("Sections saved:", len(report_sections_clean))
print("Characters:", len(report_document))

RadiologyInfo report document saved successfully.
Path: C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\documents\radiologyinfo_chest_xray_report.txt
Sections saved: 8
Characters: 5318


In [49]:
source_documents = [
    {
        "source_id": "chestxray14_dataset",
        "source_title": "NIH ChestX-ray14 Dataset",
        "source_url": "https://pmc.ncbi.nlm.nih.gov/articles/PMC9131331/",
        "sections": source_sections
    },
    {
        "source_id": "radiologyinfo_chest_xray",
        "source_title": "Chest X-ray (Radiography)",
        "source_url": "https://www.radiologyinfo.org/en/info/chestrad",
        "sections": radiology_sections
    },
    {
        "source_id": "radiologyinfo_chest_xray_report",
        "source_title": "How to Read Your Chest X-ray Report",
        "source_url": "https://www.radiologyinfo.org/en/info/article-chest-xray-report",
        "sections": report_sections_clean
    }
]

semantic_chunks = []

for source in source_documents:
    for section in source["sections"]:
        semantic_chunks.append({
            "source_id": source["source_id"],
            "source_title": source["source_title"],
            "source_url": source["source_url"],
            "section": section["section"],
            "text": section["text"]
        })

print("Total semantic chunks:", len(semantic_chunks))

for i, chunk in enumerate(semantic_chunks, start=1):
    print(
        f"{i:02d}. "
        f"{chunk['source_id']} | "
        f"{chunk['section']} | "
        f"{len(chunk['text'])} chars"
    )

Total semantic chunks: 22
01. chestxray14_dataset | Background | 686 chars
02. chestxray14_dataset | Methods | 476 chars
03. chestxray14_dataset | Results | 542 chars
04. chestxray14_dataset | Conclusions | 181 chars
05. radiologyinfo_chest_xray | What is a chest X-ray? | 437 chars
06. radiologyinfo_chest_xray | What are some common uses of the procedure? | 526 chars
07. radiologyinfo_chest_xray | How should I prepare? | 627 chars
08. radiologyinfo_chest_xray | What does the equipment look like? | 700 chars
09. radiologyinfo_chest_xray | How does the procedure work? | 996 chars
10. radiologyinfo_chest_xray | How is the procedure performed? | 1157 chars
11. radiologyinfo_chest_xray | What will I experience during and after the procedure? | 461 chars
12. radiologyinfo_chest_xray | Who interprets the results and how do I get them? | 693 chars
13. radiologyinfo_chest_xray | What are the benefits vs. risks? | 1446 chars
14. radiologyinfo_chest_xray | What are the limitations of chest x-ray?

In [50]:
for i, chunk in enumerate(semantic_chunks):
    chunk["chunk_id"] = f"rag_{i+1:03d}"
    chunk["chunk_index"] = i

print("Chunk IDs assigned:", len(semantic_chunks))

for chunk in semantic_chunks[:5]:
    print(
        chunk["chunk_id"],
        "→",
        chunk["source_id"],
        "→",
        chunk["section"]
    )
    

Chunk IDs assigned: 22
rag_001 → chestxray14_dataset → Background
rag_002 → chestxray14_dataset → Methods
rag_003 → chestxray14_dataset → Results
rag_004 → chestxray14_dataset → Conclusions
rag_005 → radiologyinfo_chest_xray → What is a chest X-ray?


In [51]:
unified_chunks_path = CHUNKS_DIR / "rag_chunks.json"

with open(unified_chunks_path, "w", encoding="utf-8") as f:
    json.dump(semantic_chunks, f, indent=2, ensure_ascii=False)

print("Unified RAG chunk corpus saved successfully.")
print("Path:", unified_chunks_path)
print("Chunks saved:", len(semantic_chunks))
print("Total characters:", sum(len(chunk["text"]) for chunk in semantic_chunks))

Unified RAG chunk corpus saved successfully.
Path: C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\chunks\rag_chunks.json
Chunks saved: 22
Total characters: 14629


In [52]:
texts = [chunk["text"] for chunk in semantic_chunks]

embeddings = embedding_model.encode(
    texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Embeddings shape:", embeddings.shape)
print("Embedding dtype:", embeddings.dtype)

Batches: 100%|██████████| 1/1 [00:00<00:00,  4.42it/s]

Embeddings shape: (22, 384)
Embedding dtype: float32


In [53]:
embedding_dim = embeddings.shape[1]

index = faiss.IndexFlatIP(embedding_dim)
index.add(embeddings)

print("FAISS index rebuilt successfully.")
print("Embedding dimension:", embedding_dim)
print("Vectors stored:", index.ntotal)

FAISS index rebuilt successfully.
Embedding dimension: 384
Vectors stored: 22


In [54]:
vector_store_path = VECTOR_STORE_DIR / "rag.index"

faiss.write_index(index, str(vector_store_path))

print("Unified FAISS index saved successfully.")
print("Path:", vector_store_path)
print("Vectors stored:", index.ntotal)
print("File size (KB):", round(vector_store_path.stat().st_size / 1024, 2))

Unified FAISS index saved successfully.
Path: C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\vector_store\rag.index
Vectors stored: 22
File size (KB): 33.04


In [55]:
rag_metadata = [
    {
        "faiss_index": chunk["chunk_index"],
        "chunk_id": chunk["chunk_id"],
        "source_id": chunk["source_id"],
        "source_title": chunk["source_title"],
        "source_url": chunk["source_url"],
        "section": chunk["section"]
    }
    for chunk in semantic_chunks
]

rag_metadata_path = METADATA_DIR / "rag_metadata.json"

with open(rag_metadata_path, "w", encoding="utf-8") as f:
    json.dump(rag_metadata, f, indent=2, ensure_ascii=False)

print("Unified RAG metadata saved successfully.")
print("Path:", rag_metadata_path)
print("Records:", len(rag_metadata))

print("\nFirst 5 records:")
for record in rag_metadata[:5]:
    print(
        f"{record['faiss_index']} → "
        f"{record['chunk_id']} → "
        f"{record['source_id']} → "
        f"{record['section']}"
    )

Unified RAG metadata saved successfully.
Path: C:\Users\yuvan\Desktop\ML_projects\Chest_Xray_AI\rag\metadata\rag_metadata.json
Records: 22

First 5 records:
0 → rag_001 → chestxray14_dataset → Background
1 → rag_002 → chestxray14_dataset → Methods
2 → rag_003 → chestxray14_dataset → Results
3 → rag_004 → chestxray14_dataset → Conclusions
4 → rag_005 → radiologyinfo_chest_xray → What is a chest X-ray?


In [56]:
test_queries = [
    "What is the ChestX-ray14 dataset?",
    "What is a chest X-ray used for?",
    "What does the Findings section of a radiology report contain?",
    "What is pneumothorax?",
    "What are the limitations of a chest X-ray?"
]

for query in test_queries:
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    scores, indices = index.search(query_embedding, 3)

    print("\n" + "=" * 90)
    print("QUERY:", query)
    print("=" * 90)

    for rank, (score, idx) in enumerate(
        zip(scores[0], indices[0]), start=1
    ):
        chunk = semantic_chunks[idx]

        print(
            f"\nRank {rank} | Score: {score:.4f} | "
            f"{chunk['source_id']} | {chunk['section']}"
        )
        print(chunk["text"][:350].replace("\n", " "))


QUERY: What is the ChestX-ray14 dataset?

Rank 1 | Score: 0.5667 | chestxray14_dataset | Methods
We experimented on the ChestX-ray14 dataset and followed the official training-test set split. Because the training data only had global annotations, the detection network was weakly supervised. A DenseNet with a feature pyramid structure was designed and integrated with an adaptive ViT to model inter-patch and patch-wise long-range dependencies an

Rank 2 | Score: 0.4719 | radiologyinfo_chest_xray | What are some common uses of the procedure?
The chest x-ray is performed to evaluate the lungs, heart and chest wall.  A chest x-ray is typically the first imaging test used to help diagnose symptoms such as:  breathing difficulties  a bad or persistent cough  chest pain or injury  fever  Physicians use the examination to help diagnose or monitor treatment for conditions such as:  pneumonia 

Rank 3 | Score: 0.4640 | radiologyinfo_chest_xray | What is a chest X-ray?
The chest x-ray is the most